# DMOSpeech 2: Analytic Unconditional Guidance Pilot (EXP-024)

**Idea.** Classifier-free guidance runs the teacher twice per step: once with the text and voice prompt ($v_c$) and once with neither ($v_u$). In flow matching, the best unconditional prediction is known in closed form while the input is still mostly noise. At $t = 0$ it is exactly the average mel, $\mu$:

$$v_u(x, t) = \frac{\mu - x}{1 - t}$$

The teacher in DMOSpeech 2's teacher-guided sampling only runs while the signal is weak: it hands over at Step 4 ($t = 0.076$) or Step 8 ($t = 0.29$). So **Analytic Unconditional Guidance (AUG)** replaces the network's unconditional pass at Steps 0..K with this formula, and keeps the conditional pass and the guidance strength. Each of those steps costs **one pass instead of two**. $\mu$ is computed once from 20 calibration cases of other speakers. **No model is trained: the released teacher and student are used unchanged.**

| Output | What it is | Passes |
|---|---|---:|
| W4_K2, W4_K4 | Step-4 handover, closed form at Steps 0-2 or 0-4 | 10, 8 (released: 13) |
| W8_K2, W8_K4 | 0.25 reference, closed form at Steps 0-2 or 0-4 | 18, 16 (released: 21) |

**What runs.**
- **Calibration:** $\mu$ from 20 speakers the pilot does not use.
- **Diagnostic and quality:** 40 development speakers x clean and white noise 15 dB (80 cases). How far the formula is from the network at Steps 0-8, how much AUG moves the teacher's handover estimate, and the AUG outputs scored against the stored outputs of the released teacher for the same cases and seeds.
- **Pitch variety:** CVf0 of the four AUG outputs on 10 speakers x 20 seeds.
- **Timing:** T4 time of the teacher with and without AUG on 10 cases.

**Time.** About 1 to 1.5 hours on a T4 (an estimate, not a measurement). Results are saved to Google Drive after every case, so a disconnected run resumes.

**Status.** A development pilot: it decides whether a frozen test on new speakers is worth running, and cannot support a claim. The go/no-go rule is frozen in `preregistration.json`. The runner refuses to start if any of its code, the preregistration or the stored input files have changed, if the GPU is not a T4, or if the smoke test did not reproduce the stored scores.

---
### Instructions
1. Set the runtime to **T4 GPU** (`Runtime > Change runtime type`).
2. **Runtime > Run all.** The smoke test must pass before the full run can start.
3. If the browser disconnects, reconnect and run **Step 4b**. If the machine resets, re-run Steps 1 to 2b, then Step 4: it continues from the last saved case.
4. When it finishes, Step 6 downloads `DMOSpeech2_EXP024_Analytic_Guidance_Pilot_Results.zip` (also saved to Drive): send that file.

### Step 1: Check the GPU (Tesla T4 required)

In [ ]:
# Checked with nvidia-smi, not PyTorch: the notebook itself then keeps no PyTorch or CUDA state in RAM,
# so the runner (which loads two TTS models, Whisper large-v3, WavLM and UTMOS) has all of it.
import subprocess
q = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
assert q.returncode == 0 and q.stdout.strip(), 'No GPU found. Switch the runtime to T4 GPU (Runtime > Change runtime type).'
gpu_name = q.stdout.strip().splitlines()[0]
print('GPU:', gpu_name)
assert 'T4' in gpu_name, f'Expected a Tesla T4 (found {gpu_name}). Student noise depends on the GPU, so the stored labels would not reproduce.'


### Step 2: Clone the Repositories and Install Dependencies

In [ ]:
import os, subprocess, hashlib, sys
THESIS_BRANCH = 'rhnl-3'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git", '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

for needed in ('code/colab/run_analytic_guidance_pilot.py', 'research/experiments/EXP-024-analytic-unconditional-guidance-pilot/preregistration.json',
               'research/experiments/EXP-018-swsr-handover-prelim/results/steps1to8.csv'):
    if not os.path.exists(f'/content/Final-Thesis-Setup/{needed}'):
        raise FileNotFoundError(f'{needed} is not on branch {THESIS_BRANCH}. Push it to GitHub first.')

DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}

!pip install -q "numpy==2.2.3" "pandas==2.2.3" audioop-lts
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub
print('Code and packages ready.')

### Step 2b: Load the Models and Dataset (Saved in Google Drive)
The first run downloads the checkpoints and dataset, then saves a copy to `MyDrive/dmospeech2_cache/`. Later runs copy them from there, so nothing is downloaded again. The checkpoint SHA-256 hashes are checked every time, and a damaged copy is replaced automatically. If Drive is full, the run continues without saving a copy.

In [ ]:
# Models and dataset are kept once in Google Drive and reused by later runs.
import os, shutil, tarfile, hashlib, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.chdir('/content/Shonget-DMOSpeech2')

CACHE = '/content/drive/MyDrive/dmospeech2_cache'
TMP_TAR = '/content/seed-tts-eval-en.tar'
os.makedirs(f'{CACHE}/ckpts', exist_ok=True)
os.makedirs('ckpts', exist_ok=True)
BASE_URL = 'https://huggingface.co/yl4579/DMOSpeech2/resolve/main'
CKPTS = {'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
         'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'}


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()


def save_to_cache(local, cached):
    # The cache is optional: a full Drive must not stop the experiment.
    try:
        shutil.copy(local, cached + '.tmp')
        os.replace(cached + '.tmp', cached)
        print(f'Saved to the Drive cache: {os.path.basename(cached)}')
    except OSError as e:
        print(f'Could not save {os.path.basename(cached)} to Drive ({e}). Continuing without the cache.')
        if os.path.exists(cached + '.tmp'):
            os.remove(cached + '.tmp')


for name, digest in CKPTS.items():
    local, cached = f'ckpts/{name}', f'{CACHE}/ckpts/{name}'
    ok = os.path.exists(local) and sha256(local) == digest
    if not ok and os.path.exists(cached):
        print(f'{name}: copying from the Drive cache')
        shutil.copy(cached, local)
        ok = sha256(local) == digest
        if not ok:
            print(f'{name}: the Drive copy is damaged, so it will be downloaded again')
            os.remove(cached)
            os.remove(local)
    if not ok:
        print(f'{name}: downloading')
        subprocess.run(['wget', '-q', '-c', f'{BASE_URL}/{name}', '-O', local], check=True)
        assert sha256(local) == digest, f'{name}: SHA-256 mismatch (delete ckpts/{name} and re-run this step)'
    print(f'{name}: SHA-256 verified')
    if not os.path.exists(cached):
        save_to_cache(local, cached)

DATA, ARCHIVE = 'data/seed-tts-eval', f'{CACHE}/seed-tts-eval-en.tar'
PROBE = f'{DATA}/en/prompt-wavs/common_voice_en_10933823.wav'


def have_data():
    return os.path.exists(PROBE) and os.path.getsize(PROBE) >= 1000 and os.path.exists(f'{DATA}/en/meta.lst')


if not have_data() and os.path.exists(ARCHIVE):
    print('Dataset: unpacking from the Drive cache')
    shutil.rmtree(DATA, ignore_errors=True)
    os.makedirs('data', exist_ok=True)
    try:
        with tarfile.open(ARCHIVE) as tar:
            tar.extractall('data')
    except (tarfile.TarError, OSError, EOFError) as e:
        print(f'The Drive copy of the dataset could not be read ({e}).')
    if not have_data():
        print('Dataset: the Drive copy is damaged, so it will be downloaded again')
        os.remove(ARCHIVE)
if not have_data():
    print('Dataset: downloading')
    shutil.rmtree(DATA, ignore_errors=True)
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir=DATA)
    assert have_data(), 'The dataset download is incomplete. Re-run this step.'
if not os.path.exists(ARCHIVE):
    print('Dataset: saving one copy to the Drive cache')
    with tarfile.open(TMP_TAR, 'w') as tar:
        tar.add(f'{DATA}/en', arcname='seed-tts-eval/en')
    save_to_cache(TMP_TAR, ARCHIVE)
    os.remove(TMP_TAR)
print('Models and dataset ready.')

### Step 3: Smoke Test (2 speakers, a few minutes)
If it stops with exit code -9, the machine ran out of RAM: use **Runtime > Disconnect and delete runtime**, then **Run all**.
Calibrates $\mu$ on 2 cases, runs every block for 2 development speakers (clean and white noise 15 dB), and does one short pitch-variety pair and one timing case. It checks three things:
- **Stored scores:** this runner reproduces the stored scores of the released Step-4 and Step-8 handovers (EXP-018).
- **Teacher loop:** its own teacher loop matches the released guided teacher.
- **AUG switched off:** with the closed form switched off, the AUG loop is the released teacher.

**If any check fails, it stops here and the full run will refuse to start.** Writes to a separate `smoke/` folder.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_analytic_guidance_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp024_analytic_guidance_pilot'
LOG = f'{PERSIST}/exp024_analytic_guidance_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

def ram_gb():
    info = dict(l.split(':', 1) for l in open('/proc/meminfo') if ':' in l)
    return int(info['MemTotal'].split()[0]) / 2**20, int(info['MemAvailable'].split()[0]) / 2**20
total, avail = ram_gb()
print(f'RAM: {avail:.1f} GB available of {total:.1f} GB')
if avail < 10:
    print('WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.')
LEAN = {'MALLOC_ARENA_MAX': '2'}   # fewer allocator arenas while the large models load; results are unaffected

proc = subprocess.Popen([sys.executable, '-u', RUNNER, 'smoke', '--out', OUT, '--dmo_root', DMO],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, **LEAN, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
code = proc.wait()
if code == -9:
    raise SystemExit('The smoke test was killed (exit code -9): the machine ran out of RAM. '
                     'Use Runtime > Disconnect and delete runtime, then Run all again.')
assert code == 0, f'Smoke test failed with code {code}'

import pandas as pd
print(pd.read_csv(f'{OUT}/smoke/pilot.csv')[['utt_id', 'condition', 'prompt_snr', 'e_0', 'e_4', 'gap0_seed', 'd_slow_K4_4', 'y_slow_4', 'wer_W4', 'wer_W4_K4', 'wer_W8', 'wer_W8_K4', 'teacher_loop_max_abs_diff', 'aug_off_max_abs_diff', 'case_seconds']].to_string())

### Step 4: Full Run + Analysis (about 1 to 1.5 hours, estimated)
Runs in the background and writes its output to `MyDrive/dmospeech2_results/exp024_analytic_guidance_pilot_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running: reconnect and run **Step 4b**. If the machine resets, re-run Steps 1 to 2b, then this cell.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_analytic_guidance_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp024_analytic_guidance_pilot'
LOG = f'{PERSIST}/exp024_analytic_guidance_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

def ram_gb():
    info = dict(l.split(':', 1) for l in open('/proc/meminfo') if ':' in l)
    return int(info['MemTotal'].split()[0]) / 2**20, int(info['MemAvailable'].split()[0]) / 2**20
total, avail = ram_gb()
print(f'RAM: {avail:.1f} GB available of {total:.1f} GB')
if avail < 10:
    print('WARNING: little RAM is free. If the job is killed (exit code -9), use Runtime > Disconnect and delete runtime, then Run all.')
LEAN = {'MALLOC_ARENA_MAX': '2'}   # fewer allocator arenas while the large models load; results are unaffected

import time, re
cmd = f'{sys.executable} -u "{RUNNER}" run --out "{OUT}" --dmo_root {DMO}'
running = subprocess.run(['pgrep', '-f', 'run_analytic_guidance_pilot.py'], capture_output=True, text=True).stdout.split()
if running:
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f'A job is already running (pid {pid}); monitoring it instead of starting a new one.')
else:
    proc = subprocess.Popen(f'nohup {cmd} >> "{LOG}" 2>&1', shell=True, env={**os.environ, **LEAN})
    alive = lambda: proc.poll() is None
    print(f'Started (pid {proc.pid}). Full log: {LOG}')

last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(LOG, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l or 'SystemExit' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
if not running and proc.returncode in (-9, 137):
    print('The job was killed for lack of RAM. Use Runtime > Disconnect and delete runtime, then run Steps 1 to 2b and this cell: '
          'it resumes from the last saved case.')
for line in open(LOG, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 4b: Check Progress After a Reconnect
Shows whether the background job is running and how many cases are saved. Does **not** start a second job.

In [ ]:
import os, re, subprocess
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp024_analytic_guidance_pilot'
LOG = f'{PERSIST}/exp024_analytic_guidance_pilot_log.txt'
running = subprocess.run(['pgrep', '-f', 'run_analytic_guidance_pilot.py'], capture_output=True, text=True).stdout.split()
print('Job running:', 'yes (pid ' + ', '.join(running) + ')' if running else 'no')
if os.path.exists(LOG):
    lines = open(LOG, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print('Latest progress:', prog[-1] if prog else '(none yet)')
for name, total in (('pilot.csv', 80), ('diversity.csv', 10), ('timing.csv', 10)):
    path = os.path.join(OUT, name)
    n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
    print(f'{name}: {n} of {total} saved')

### Step 5: View the Report
If the run stopped early, add `'--allow_partial'` to the command for an interim look (the report is then marked PARTIAL).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_analytic_guidance_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp024_analytic_guidance_pilot'
LOG = f'{PERSIST}/exp024_analytic_guidance_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

from IPython.display import Markdown, display
rep = f'{OUT}/AUG_REPORT.md'
if not os.path.exists(rep):
    subprocess.run([sys.executable, RUNNER, 'analyze', '--out', OUT], check=True)
display(Markdown(open(rep, encoding='utf-8').read()))

### Step 6: Zip All Outputs and Download
Packs the CSV files, preregistration, results JSON, report and runner scripts into one ZIP, also saved to Drive. Audio is not included.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_analytic_guidance_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp024_analytic_guidance_pilot'
LOG = f'{PERSIST}/exp024_analytic_guidance_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

import zipfile, datetime, torch
zip_name = 'DMOSpeech2_EXP024_Analytic_Guidance_Pilot_Results.zip'
git = lambda *a: subprocess.run(['git', *a, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(f'{OUT}/RUN_INFO.txt', 'w') as f:
    f.write(f"dmospeech2_commit: {git()}\n")
    f.write(f"thesis_commit: {git('-C', THESIS)}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base, dirs, names in os.walk(OUT):
        for fname in names:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(base, fname)
            zf.write(full, os.path.relpath(full, OUT))
    for script in ('run_analytic_guidance_pilot.py', 'run_selective_handover_prelim.py', 'run_grid32_prelim.py', 'run_noise_guidance_prelim.py',
                   'run_endpoint_extrapolation_prelim.py', 'run_consistency_gate_prelim.py', 'run_followup_tests.py'):
        zf.write(f'{THESIS}/code/colab/{script}', f'code/{script}')
print(f'Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)')
!cp "{zip_name}" "{PERSIST}/"
print(f'Also saved to Google Drive: {PERSIST}/{zip_name}')
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f'Download manually from the Files panel: {DMO}/{zip_name}')